# L3c Example: Recursive Implementation of Fibonacci Sequence Calculation
In this example, we illustrate recursion concepts by benchmarking three implementations of the Fibonacci sequence computation using [the BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl).

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
> * __Measure instead of guessing:__ Time each version of the same calculation at the same problem size. Check that each version returns the right answer before timing it, so the comparison is between versions that are all correct.
> * __Find the repeated work:__ Explain why a plain recursion solves the same subproblem more than once instead of reusing the answer it already has. Say why that repetition, and not the cost of calling a function, is what makes the recursion slower than the loop.
> * __Use memoization:__ Add the one check that returns a stored subproblem instead of computing it again, and explain why that single line changes the running time. Explain what a benchmark has to do to measure such a cache from empty, rather than timing one that earlier samples already filled.

We expect the memoized version to beat the plain recursion by a wide margin, because memoization removes repeated work instead of just making it faster. Let's see whether the measurements agree. Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading the packages and files we need.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

The three Fibonacci implementations we benchmark come from [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl); see [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/). This example also uses `Base`, [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/), and [the BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl).

### Implementations
The three versions we benchmark are in [the `Recursion.jl` file](../../../code/src/Recursion.jl) of the course package, which [`Include.jl`](Include.jl) loads for us.

* __Vanilla loop-based implementation:__ The `fibonacci(n::Int64)::Dict{Int64, Int64}` function fills a dictionary from `0` to `n` with a single for-loop.
* __Standard recursive implementation:__ The `fibonacci!(n::Int64, series::Dict{Int64, Int64})::Int64` function computes $F_{n}$ straight from the recurrence. It stores each index it visits in `series`, but it does not check `series` before recursing, so the same subproblems are recomputed many times.
* __Memoized recursive implementation:__ The `memoization_fibonacci!(n::Int64, series::Dict{Int64, Int64})::Int64` function is the same recursion with one extra line: it returns a stored value when it finds one, so each subproblem is computed exactly once.

The loop implementation returns the whole dictionary. Both recursive implementations write every index they visit into the `series` dictionary they are handed and return $F_{n}$ as an `Int64`.


### Constants
Let's set the constants this example uses. The `correct_fibonacci_sequence` constant holds the values we check each implementation against, and `benchmark_index` fixes the problem size, which every case uses so the three timings can be compared.

In [ ]:
correct_fibonacci_sequence = Dict(0 => 0, 1 => 1, 2 => 1, 3 => 2, 4 => 3, 5 => 5, 6 => 8, 7 => 13,
                                  8 => 21, 9 => 34, 10 => 55, 11 => 89, 12 => 144, 13 => 233,
                                  14 => 377, 15 => 610); # F0 through F15, so 16 known values
benchmark_index = 25; # every case below is benchmarked at this same n, so the timings are comparable

___

## Case 1: Test the for loop implementation of Fibonacci computation
In this case, we test and then time the for-loop implementation of the Fibonacci calculation. This is the baseline for the two recursive versions, since a runtime only means something next to another runtime.

Let's use the [BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl) to compute the average time required to calculate the sequence $F_{0},\dots,F_{n}$ using the vanilla implementation of the `fibonacci` function (for-loop-based implementation). However, before we benchmark the for loop implementation, let's check that it is correct by [using the `@test` macro exported by the `Test.jl` package](https://docs.julialang.org/en/v1/stdlib/Test/).

In [ ]:
let

    # initialize -
    number_of_test_terms = 15; # the reference dictionary holds F0 through F15
    my_computed_sequence = fibonacci(number_of_test_terms);

    for i ∈ 0:number_of_test_terms
        @test my_computed_sequence[i] == correct_fibonacci_sequence[i];
    end
end

Now that we have verified correctness, let's benchmark the for-loop implementation of the Fibonacci sequence calculation.

> __How we time each version:__ The [BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl) exports the [@benchmarkable macro](https://juliaci.github.io/BenchmarkTools.jl/stable/reference/#BenchmarkTools.@benchmarkable-Tuple), which builds a benchmark for a call without running it. Running that benchmark executes the call many times and returns a trial holding the runtime and memory profile of the run. We fix `samples` and `evals` explicitly rather than calling `tune!`, so every case runs at the same sample count and the same problem size. The reason `evals = 1` matters shows up in Case 3.

One difference stays. This case builds its own dictionary inside the timed call, where Cases 2 and 3 are handed an empty one before the clock starts, so read the runtimes against each other but expect the allocation counts to differ by that dictionary. The `result_basal::BenchmarkTools.Trial` variable holds the timing and allocation results of the run:

In [ ]:
result_basal = let
    test_run_basal = @benchmarkable fibonacci($(benchmark_index));
    result_basal = run(test_run_basal; samples = 200, evals = 1)
end

___

## Case 2: Test the recursive implementation of the Fibonacci computation
In this case, we test and then time the plain recursive implementation at the same problem size used in Case 1. The recursion states the Fibonacci rule almost exactly as it is written in mathematics, which makes its runtime surprising.

Next, let's benchmark a recursive implementation. The `fibonacci!(n::Int64, series::Dict{Int64, Int64})::Int64` function is a mutating recursive function that computes the sequence $F_{0},\dots, F_{n}$ for a given $n$. The recursive sequence is stored in the `series::Dict{Int64, Int64}` argument. This takes advantage of [the mutating function behavior](https://docs.julialang.org/en/v1/manual/functions/#man-argument-passing) in Julia, which allows us to update the dictionary in place without returning a new dictionary.

Let's verify that the recursive implementation is correct by checking that it computes the Fibonacci sequence correctly for $F_{0},\dots,F_{n}$, where $n$ is the largest index in the `correct_fibonacci_sequence` dictionary.

In [ ]:
let

    # initialize -
    number_of_test_terms = 15; # the reference dictionary holds F0 through F15
    my_computed_sequence = Dict{Int64, Int64}(); # initialize an empty dictionary
    fibonacci!(number_of_test_terms, my_computed_sequence); # mutates the dictionary in place; the returned Fn is ignored here

    # verify correctness - for terms 0 ... number_of_test_terms
    for i ∈ 0:number_of_test_terms
        @test my_computed_sequence[i] == correct_fibonacci_sequence[i];
    end
end

The tests pass, so the recursion computes the right values. How does it perform relative to the loop? The `result_recursive::BenchmarkTools.Trial` variable holds that measurement, taken at the same problem size and sample count as Case 1:

In [ ]:
result_recursive = let
    test_run_recursive = @benchmarkable fibonacci!($(benchmark_index), series) setup=(series = Dict{Int,Int}())
    result_recursive = run(test_run_recursive; samples = 200, evals = 1)
end

At the same `benchmark_index`, the plain recursion is __far slower__ than the loop. The cause is not the cost of a function call. It is that [the `fibonacci!(...)` function](../../../code/src/Recursion.jl) never checks `series` before recursing, so it computes $F_{n-1}$ and $F_{n-2}$ again at every level, and the number of calls grows exponentially in `n`. So recursion is not automatically slower than iteration, but a recursion that recomputes its own subproblems is.

The call tree is the reason. Each node is a call, and the plain recursion rebuilds the whole shaded subtree every time it needs a value it has already computed. Memoization means answering the second visit to a node from a table instead.

<div>
    <center>
      <img
        src="figs/Fig-Fibonacci-Recursive.svg"
        alt="Call tree for the recursive Fibonacci computation, showing repeated subtrees"
        height="400"
        width="800"
      />
    </center>
  </div>

___

## Case 3: Test the recursive implementation of the Fibonacci computation with memoization
In this case, we test and then time the same recursion again, with memoization added so each subproblem is computed only once. Case 2 measured what the repeated work costs, and this case measures what we get back by removing it.

Finally, let's benchmark a recursive Fibonacci function that uses memoization. The `memoization_fibonacci!(n::Int64, series::Dict{Int64, Int64})::Int64` implementation is a mutating recursive function that uses memoization to speed up the computation of the sequence $F_{0},\dots, F_{n}$ for a given $n$. The recursive sequence is stored in the `series::Dict{Int64, Int64}` argument.

> __Why is there no memory cost here?__ Memoization normally trades memory for time: you keep a table of answers you would otherwise recompute. This example does not show that trade. Our plain recursion was already handed a dictionary to record every index it visits, so both recursive versions finish with the same `series` holding $F_{0},\dots,F_{n}$, and the memoized version adds one lookup rather than one table.

First, does this implementation do what we expect? Let's verify that it computes the Fibonacci sequence correctly for $F_{0},\dots,F_{n}$, where $n$ is the largest index in the `correct_fibonacci_sequence` dictionary.

In [ ]:
let

    # initialize -
    number_of_test_terms = 15; # the reference dictionary holds F0 through F15
    my_computed_sequence = Dict{Int64, Int64}(); # initialize an empty dictionary
    memoization_fibonacci!(number_of_test_terms, my_computed_sequence); # mutates the dictionary in place; the returned Fn is ignored here

    # verify correctness - for terms 0 ... number_of_test_terms
    for i ∈ 0:number_of_test_terms
        @test my_computed_sequence[i] == correct_fibonacci_sequence[i];
    end
end

Does memoization change the runtime and allocation profile of the recursive implementation? Let's benchmark the memoized recursion at the same problem size and sample count as Case 2. The `result_recursive_memo::BenchmarkTools.Trial` variable holds that measurement.

> __Why both benchmark settings matter:__
>
> An __evaluation__ is one timed run of the code. A __sample__ is one measurement, made from `evals` evaluations, and this benchmark collects up to 200 samples.
>
> Each sample runs the `setup=` code once, outside the part being timed, so every sample starts from its own empty dictionary. If the dictionary were built once outside the benchmark instead, the samples would share it: the first would fill it, and the rest would find every answer already stored.
>
> Setting `evals = 1` makes the same thing true inside one sample. With `evals = 5`, the setup would still run once while the timed code ran five times, so only the first of those five would start from an empty dictionary. Case 2 uses the same `setup=` and the same `evals = 1`.

With both settings in place, the `result_recursive_memo::BenchmarkTools.Trial` variable holds the time for one memoized run that starts from an empty dictionary:

In [ ]:
result_recursive_memo = let
    test_run_recursive_memo = @benchmarkable memoization_fibonacci!($(benchmark_index), series) setup=(series = Dict{Int,Int}())
    result_recursive_memo = run(test_run_recursive_memo; samples = 200, evals = 1)
end

___

## Summary
Three implementations of one calculation, benchmarked at the same problem size, separate the cost of recursion from the cost of repeated work.

> __Key Takeaways:__
>
> * __Recursion is not slow by itself:__ The plain recursive version was slower than the loop because its call tree solves the same subproblems over and over, not because a function call is expensive. So the fix is to remove the repeated work rather than to abandon recursion.
> * __Memoization solves each subproblem once:__ Storing a subproblem the first time it is solved lets every later call for it return the stored value instead of rebuilding that subtree, which makes the cached version faster by orders of magnitude. It usually costs memory to save that time, though not here, because the plain recursion was already filling the same dictionary and simply never read it.
> * __A benchmark is only as good as its setup:__ Timing two versions at different problem sizes, or reusing a cache that earlier samples filled, measures something other than what you meant to measure. Holding the problem size fixed across all three cases and rebuilding the cache for every sample is what makes these timings comparable.

Choose the algorithm from the structure of the problem, and remember that a clean recursive statement sometimes needs memoization before it is practical.
___